# Camada Silver — V-Commerce CRM 360

**Módulo 1 · Engenharia de Dados · Arquitetura Medalhão**

---

## Visão Geral

A camada Silver é responsável por transformar os dados brutos ingeridos na Bronze em tabelas limpas, padronizadas e semanticamente organizadas segundo a **modelagem dimensional** (estrela). Cada notebook desta camada lê de `bronze.*`, aplica as transformações necessárias e grava em `silver.*`.

Os princípios aplicados em todas as tabelas Silver são:

- **Deduplicação** — reprocessamentos da Bronze (modo `append`) são neutralizados via `row_number()` sobre a chave primária ordenada por `timestamp_ingestion`
- **Tipagem explícita** — colunas lidas como `string` na Bronze recebem o tipo correto (`timestamp`, `decimal`, `boolean`, `int`)
- **Sanitização textual** — valores categóricos inconsistentes (variações de case, abreviações, typos) são normalizados para um conjunto canônico
- **Colunas derivadas** — atributos calculáveis a partir dos dados brutos são materializados para evitar recomputação nas camadas superiores
- **Idempotência** — todas as escritas usam `mode=overwrite`, garantindo que reexecutar o notebook produza sempre o mesmo resultado
- **Rastreabilidade** — `timestamp_ingestion` é recriado com o instante da carga Silver

---

## Tabelas produzidas neste notebook

### Tabelas Fato

| Tabela Silver | Origem Bronze | Descrição |
|---|---|---|
| `ft_tickets_suporte` | `suporte_tickets` | Tickets do SAC limpos, tipados e com colunas derivadas |
| `ft_pedidos` | `pedidos` | Histórico de compras com valor total e ano_mes derivados |
| `ft_avaliacoes` | `avaliacoes` | Avaliações pós-compra com categoria NPS derivada |
| `ft_clickstream` | `clickstream` | Eventos de navegação tipados e normalizados |

### Dimensões

| Tabela Silver | Origem Bronze | Descrição |
|---|---|---|
| `dim_clientes` | `clientes` | Perfis cadastrais com idade, nome completo e região derivados |
| `dim_produtos` | `catalogo_produtos` | Catálogo de produtos com categoria normalizada |
| `dim_tipos_problema` | `suporte_tickets` | Tipos de problema canônicos com categoria de negócio |
| `dim_agentes_suporte` | `suporte_tickets` | Agentes com métricas agregadas de desempenho |
| `dim_status_pedido` | `pedidos` | Status de pedido normalizados |
| `dim_categorias_produto` | `catalogo_produtos` | Categorias de produto normalizadas |

---

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window
catalogo       = 'vcommerce_catalog'
bronze_schema  = 'vcommerce_bronze'
silver_schema  = 'vcommerce_silver'

spark.sql(f'USE CATALOG {catalogo}')
spark.sql(f'CREATE SCHEMA IF NOT EXISTS {silver_schema}')
spark.sql(f'USE SCHEMA {silver_schema}')

print(f'Catálogo : {catalogo}')
print(f'Schema   : {silver_schema}')

---

## Tratamento: `ft_tickets_suporte`

**Origem:** `bronze.suporte_tickets`  
**Destino:** `silver.ft_tickets_suporte`, `silver.dim_tipos_problema`, `silver.dim_agentes_suporte`

### Problemas identificados na Bronze

| Coluna | Problema | Tratamento |
|---|---|---|
| `tipo_problema` | ~25 variações para 4 categorias reais (`pro`, `p3oduto`, `PRODUCT`, `DELAY`…) | Mapeamento para valores canônicos |
| `data_resolucao` | 2.072 nulos — tickets ainda abertos | Mantidos como `null`; `resolvido = false` |
| `nota_avaliacao` | 2.072 nulos — sem avaliação para tickets abertos | Mantidos como `null` |
| `tempo_resolucao_horas` | 2.072 nulos — tickets não resolvidos | Mantidos como `null` |
| Duplicatas | Possíveis reprocessamentos da Bronze (`mode=append`) | Deduplicação por `ticket_id` |
| Datas | Formato ISO com timezone (`2023-01-03T05:13:00.000Z`) | Cast para `timestamp` |

### Colunas derivadas

| Coluna | Lógica |
|---|---|
| `resolvido` | `data_resolucao IS NOT NULL` |
| `hora_abertura` | `HOUR(data_abertura)` |
| `dia_semana_abertura` | `DAYOFWEEK(data_abertura)` |

In [0]:
# ─── Leitura da Bronze ────────────────────────────────────────────────────────
# Usa a última ingestão de cada ticket (maior timestamp_ingestion) para
# garantir que reprocessamentos da Bronze não gerem duplicatas na Silver.

df_raw = spark.table(f'{bronze_schema}.suporte_tickets')

window_dedup = Window.partitionBy('ticket_id').orderBy(F.desc('timestamp_ingestion'))

df_dedup = (
    df_raw
    .withColumn('_rank', F.row_number().over(window_dedup))
    .filter(F.col('_rank') == 1)
    .drop('_rank', 'timestamp_ingestion')   # será recriado com o instante desta carga
)

total_raw   = df_raw.count()
total_dedup = df_dedup.count()
print(f'Registros na Bronze  : {total_raw:,}')
print(f'Após deduplicação    : {total_dedup:,}')
print(f'Duplicatas removidas : {total_raw - total_dedup:,}')

In [0]:
# ─── Mapeamento canônico de tipo_problema ────────────────────────────────────
# Na Bronze foram encontradas 4 categorias reais fragmentadas em ~25 variações:
#
#   Entrega   - Entrega, ENTREGA, entrega, 3ntrega, entr, del, DELAY, delay
#   Reembolso - Reembolso, REEMBOLSO, reembolso, REFUND, ref, reemb, r3embolso
#   Produto   - Produto, PRODUTO, produto, pro, prod, p3oduto, PRODUCT, product
#   Pagamento - Pagamento, PAGAMENTO, pagamento, pag, pay, PAY, PAYMENT, payment,
#               p4gamento
#
# Estratégia: normaliza para lowercase e aplica mapeamento por prefixo/palavra-chave.
# Valores não mapeados são marcados como 'Outro' para investigação futura.

tipo_map = {
    # Entrega
    'entrega'  : 'Entrega',  '3ntrega' : 'Entrega',  'entr'  : 'Entrega',
    'del'      : 'Entrega',  'delay'   : 'Entrega',
    # Reembolso
    'reembolso': 'Reembolso','reemb'   : 'Reembolso','r3embolso': 'Reembolso',
    'refund'   : 'Reembolso','ref'     : 'Reembolso',
    # Produto
    'produto'  : 'Produto',  'pro'     : 'Produto',  'prod'  : 'Produto',
    'p3oduto'  : 'Produto',  'product' : 'Produto',
    # Pagamento
    'pagamento': 'Pagamento','pag'     : 'Pagamento','p4gamento': 'Pagamento',
    'pay'      : 'Pagamento','payment' : 'Pagamento',
}

# Constrói expressão CASE WHEN a partir do dicionário
tipo_expr = F.col('tipo_problema')
for raw_val, canonical in tipo_map.items():
    tipo_expr = F.when(
        F.lower(F.col('tipo_problema')) == raw_val, canonical
    ).otherwise(tipo_expr)

# Valores que não bateram com nenhum mapeamento ficam como 'Outro'
known_lower = [k.lower() for k in tipo_map.keys()]
tipo_expr = F.when(
    F.lower(F.col('tipo_problema')).isin(known_lower), tipo_expr
).otherwise(F.lit('Outro'))

print('Expressão de mapeamento criada.')

In [0]:
# ─── Transformações principais ───────────────────────────────────────────────

df_silver = (
    df_dedup

    # 1. Tipos corretos para datas (ISO 8601 com timezone)
    .withColumn('data_abertura',   F.to_timestamp('data_abertura'))
    .withColumn('data_resolucao',  F.to_timestamp('data_resolucao'))

    # 2. Normalização de tipo_problema
    .withColumn('tipo_problema', tipo_expr)

    # 3. Colunas derivadas de negócio
    .withColumn(
        'resolvido',
        F.col('data_resolucao').isNotNull()
    )
    .withColumn(
        'hora_abertura',
        F.hour('data_abertura').cast('int')
    )
    .withColumn(
        'dia_semana_abertura',
        F.date_format('data_abertura', 'EEEE')   # nome completo em inglês; adaptar locale se necessário
    )

    # 4. Tipos numéricos
    .withColumn('tempo_resolucao_horas', F.col('tempo_resolucao_horas').cast('decimal(10,2)'))
    .withColumn('nota_avaliacao',        F.col('nota_avaliacao').cast('decimal(3,1)'))

    # 5. Marca temporal desta carga Silver
    .withColumn('timestamp_ingestion', F.current_timestamp())

    # 6. Ordenação de colunas conforme schema Silver acordado
    .select(
        'ticket_id',
        'id_cliente',
        'id_pedido',
        'tipo_problema',
        'data_abertura',
        'data_resolucao',
        'tempo_resolucao_horas',
        'agente_suporte',
        'nota_avaliacao',
        'resolvido',
        'hora_abertura',
        'dia_semana_abertura',
        'timestamp_ingestion',
    )
)

print('Transformações aplicadas. Schema resultante:')
df_silver.printSchema()

In [0]:
# ─── Validação pré-escrita ────────────────────────────────────────────────────

total = df_silver.count()
resolvidos     = df_silver.filter(F.col('resolvido') == True).count()
nao_resolvidos = df_silver.filter(F.col('resolvido') == False).count()
outros_tipo    = df_silver.filter(F.col('tipo_problema') == 'Outro').count()

print(f'Total de tickets       : {total:,}')
print(f'Resolvidos             : {resolvidos:,} ({resolvidos/total*100:.1f}%)')
print(f'Abertos (sem resolução): {nao_resolvidos:,} ({nao_resolvidos/total*100:.1f}%)')
print(f'Tipo "Outro" (suspeitos): {outros_tipo:,}')
print()
print('Distribuição de tipo_problema após normalização:')
df_silver.groupBy('tipo_problema').count().orderBy(F.desc('count')).show()

In [0]:
# ─── Grava silver.ft_tickets_suporte ─────────────────────────────────────────
# overwrite garante idempotência: reexecutar o notebook produz o mesmo resultado.

(
    df_silver.write
             .format('delta')
             .mode('overwrite')
             .option('overwriteSchema', 'true')
             .saveAsTable(f'{silver_schema}.ft_tickets_suporte')
)

print(f'{silver_schema}.ft_tickets_suporte gravada com {df_silver.count():,} registros.')

In [0]:
# ─── dim_tipos_problema ───────────────────────────────────────────────────────
# Dimensão com os 4 tipos canônicos e sua categoria de negócio.
#
# Categoria derivada:
#   Entrega   - Logística
#   Reembolso - Financeiro
#   Produto   - Qualidade
#   Pagamento - Financeiro
#   Outro     - Indefinido

df_dim_tipos = (
    spark.table(f'{silver_schema}.ft_tickets_suporte')
         .select('tipo_problema')
         .distinct()
         .withColumn(
             'categoria_problema',
             F.when(F.col('tipo_problema') == 'Entrega',   'Logística')
              .when(F.col('tipo_problema') == 'Reembolso', 'Financeiro')
              .when(F.col('tipo_problema') == 'Produto',   'Qualidade')
              .when(F.col('tipo_problema') == 'Pagamento', 'Financeiro')
              .otherwise('Indefinido')
         )
         .orderBy('tipo_problema')
)

df_dim_tipos.show()

(
    df_dim_tipos.write
                .format('delta')
                .mode('overwrite')
                .option('overwriteSchema', 'true')
                .saveAsTable(f'{silver_schema}.dim_tipos_problema')
)

print(f'{silver_schema}.dim_tipos_problema gravada.')

In [0]:
# ─── dim_agentes_suporte ──────────────────────────────────────────────────────
# Dimensão com métricas agregadas por agente, derivadas de ft_tickets_suporte.
#
# Métricas calculadas:
#   qtd_tickets_resolvidos  - total de tickets onde resolvido = true
#   nota_media_atendimento  - média de nota_avaliacao (ignora nulos)

df_dim_agentes = (
    spark.table(f'{silver_schema}.ft_tickets_suporte')
         .groupBy('agente_suporte')
         .agg(
             F.count(
                 F.when(F.col('resolvido') == True, 1)
             ).alias('qtd_tickets_resolvidos'),
             F.round(
                 F.avg('nota_avaliacao'), 2
             ).alias('nota_media_atendimento'),
         )
         .withColumn(
             'nota_media_atendimento',
             F.col('nota_media_atendimento').cast('decimal(4,2)')
         )
         .orderBy(F.desc('qtd_tickets_resolvidos'))
)

df_dim_agentes.show(truncate=False)

(
    df_dim_agentes.write
                  .format('delta')
                  .mode('overwrite')
                  .option('overwriteSchema', 'true')
                  .saveAsTable(f'{silver_schema}.dim_agentes_suporte')
)

print(f'{silver_schema}.dim_agentes_suporte gravada.')

---

## Tratamento: `ft_clickstream`

**Origem:** `bronze.clickstream`  
**Destino:** `silver.ft_clickstream`

### Problemas identificados na Bronze

| Coluna | Problema | Tratamento |
|---|---|---|
| `tipo_evento` | Múltiplas variações para mesmas categorias (`view`, `VISUALIZACAO`, `page_view`…) | Mapeamento para 6 valores canônicos |
| `dispositivo` | Inconsistências de case e sinônimos (`MOBILE`, `celular`, `smartphone`) | Normalização para `Desktop`, `Mobile`, `Tablet` |
| `origem_sessao` | Variações em inglês/português e case (`organic`, `ORGANICO`, `cpc`…) | Normalização para 6 canais canônicos |
| `timestamp_evento` | Formato ISO 8601 com timezone lido como `string` na Bronze | Cast para `timestamp` |
| `tempo_pagina_seg` | Possíveis valores negativos ou zero (sessões inválidas) | Substituídos por `null` |
| Duplicatas | Possíveis reprocessamentos da Bronze | Deduplicação por `id_evento` |

### Colunas derivadas

| Coluna | Lógica |
|---|---|
| `hora_evento` | `HOUR(timestamp_evento)` |
| `dia_semana_evento` | `date_format(timestamp_evento, 'EEEE')` |
| `periodo_dia` | Madrugada (0–5 h), Manhã (6–11 h), Tarde (12–17 h), Noite (18–23 h) |
| `is_conversao` | `tipo_evento == 'Compra'` |
| `ano_mes_evento` | `date_format(timestamp_evento, 'yyyy-MM')` para agregações mensais |

In [0]:
# ─── Leitura da Bronze e deduplicação ───────────────────────────────────────
# event_id é a chave primária de cada evento de navegação.
# Mantém apenas o registro mais recente de cada event_id para neutralizar
# reprocessamentos em modo append na Bronze.

df_raw_cs = spark.table(f'{bronze_schema}.clickstream')

window_dedup_cs = Window.partitionBy('id_evento').orderBy(F.desc('timestamp_ingestion'))

df_dedup_cs = (
    df_raw_cs
    .withColumn('_rank', F.row_number().over(window_dedup_cs))
    .filter(F.col('_rank') == 1)
    .drop('_rank', 'timestamp_ingestion')
)

total_raw_cs   = df_raw_cs.count()
total_dedup_cs = df_dedup_cs.count()
print(f'Registros na Bronze  : {total_raw_cs:,}')
print(f'Após deduplicação    : {total_dedup_cs:,}')
print(f'Duplicatas removidas : {total_raw_cs - total_dedup_cs:,}')

In [0]:
# ─── Mapeamento canônico de tipo_evento ────────────────────────────────────
# 6 categorias reais encontradas com múltiplas grafias na Bronze:
#
#   Visualizacao - visualizacao, VISUALIZACAO, view, VIEW, page_view, pageview
#   Clique       - clique, CLIQUE, click, CLICK
#   Carrinho     - add_carrinho, ADD_CARRINHO, add_to_cart, addtocart, adicionar_carrinho
#   Checkout     - checkout, CHECKOUT, chk
#   Compra       - compra, COMPRA, purchase, PURCHASE, buy
#   Busca        - busca, BUSCA, search, SEARCH

tipo_evento_map = {
    # Visualização
    'visualizacao' : 'Visualização',
    'view'         : 'Visualização',
    'page_view'    : 'Visualização',
    'pageview'     : 'Visualização',
    'page-view'    : 'Visualização',
    'pv'           : 'Visualização',
    'product_view' : 'Visualização',
    # Clique
    'clique' : 'Clique',
    'click'  : 'Clique',
    # Carrinho
    'add_carrinho'       : 'Carrinho',
    'add_to_cart'        : 'Carrinho',
    'addtocart'          : 'Carrinho',
    'adicionar_carrinho' : 'Carrinho',
    'adicionar'          : 'Carrinho',
    # Checkout
    'checkout' : 'Checkout',
    'chk'      : 'Checkout',
    # Compra
    'compra'   : 'Compra',
    'purchase' : 'Compra',
    'buy'      : 'Compra',
    # Busca
    'busca'  : 'Busca',
    'search' : 'Busca',
    'srch'   : 'Busca',
    # Outros eventos relevantes
    'login'         : 'Login',
    'abandon_cart'  : 'Abandono Carrinho',
}

tipo_evento_expr = F.col('tipo_evento')
for raw_val, canonical in tipo_evento_map.items():
    tipo_evento_expr = F.when(
        F.lower(F.col('tipo_evento')) == raw_val, canonical
    ).otherwise(tipo_evento_expr)

known_tipos = list(tipo_evento_map.keys())
tipo_evento_expr = F.when(
    F.lower(F.col('tipo_evento')).isin(known_tipos), tipo_evento_expr
).otherwise(F.lit('Outro'))

print('Expressão tipo_evento criada.')

In [0]:
# ─── Mapeamento canônico de dispositivo e canal ────────────────────────────

dispositivo_map = {
    # Desktop
    'desktop'    : 'Desktop',
    'computador' : 'Desktop',
    # Mobile
    'mobile'     : 'Mobile',
    'smartphone' : 'Mobile',
    'celular'    : 'Mobile',
    'mob'        : 'Mobile',
    # Tablet
    'tablet' : 'Tablet',
    'tab'    : 'Tablet',
}

dispositivo_expr = F.col('dispositivo')
for raw_val, canonical in dispositivo_map.items():
    dispositivo_expr = F.when(
        F.lower(F.col('dispositivo')) == raw_val, canonical
    ).otherwise(dispositivo_expr)

known_disp = list(dispositivo_map.keys())
dispositivo_expr = F.when(
    F.lower(F.col('dispositivo')).isin(known_disp), dispositivo_expr
).otherwise(F.lit('Outro'))

# ── Canal de tráfego ──────────────────────────────────────────────────────────
origem_sessao_map = {
    'organico'      : 'Organico',
    'organic'       : 'Organico',
    'pago'          : 'Pago',
    'paid'          : 'Pago',
    'paid_search'   : 'Pago',
    'ads'           : 'Pago',
    'social'        : 'Redes Sociais',
    'redes_sociais' : 'Redes Sociais',
    'social_media'  : 'Redes Sociais',
    'email'         : 'Email',
    'e-mail'        : 'Email',
    'direto'        : 'Direto',
    'direct'        : 'Direto',
}

origem_sessao_expr = F.col('origem_sessao')
for raw_val, canonical in origem_sessao_map.items():
    origem_sessao_expr = F.when(
        F.lower(F.col('origem_sessao')) == raw_val, canonical
    ).otherwise(origem_sessao_expr)

known_origin = list(origem_sessao_map.keys())
origem_sessao_expr = F.when(
    F.lower(F.col('origem_sessao')).isin(known_origin), origem_sessao_expr
).otherwise(F.lit('Outro'))

print('Expressões dispositivo e origem_sessao criadas.')

In [0]:
# ── Canal de marketing (agrupamento de nível superior) ───────────────────────
canal_map = {
  # Web
  'web' : 'Web',
  # App
  'app'        : 'App',
  'aplicativo' : 'App',
  # Mobile Web
  'mobile_web' : 'Mobile Web',
  'mobile web' : 'Mobile Web',
  # Extras (caso apareçam no futuro)
  'mobile'  : 'Mobile Web',
  'desktop' : 'Web',
}

canal_expr = F.col('canal')
for raw_val, canonical in canal_map.items():
    canal_expr = F.when(
        F.lower(F.col('canal')) == raw_val, canonical
    ).otherwise(canal_expr)

known_canal = list(canal_map.keys())
canal_expr = F.when(
    F.lower(F.col('canal')).isin(known_canal), canal_expr
).otherwise(F.lit('Outro'))

print('Expressoes canal criadas.')

In [0]:
# ─── Transformações principais ───────────────────────────────────────────────

df_silver_cs = (
    df_dedup_cs

    # 1. Tipo correto para o timestamp do evento
    .withColumn('data_evento', F.to_timestamp('data_evento'))

    # 2. Normalização de categóricas
    .withColumn('tipo_evento', tipo_evento_expr)
    .withColumn('dispositivo', dispositivo_expr)
    .withColumn('origem_sessao',origem_sessao_expr)
    .withColumn('canal', canal_expr)

    # 3. Tipos numéricos
    .withColumn('tempo_pagina_seg', F.col('tempo_pagina_seg').cast('int'))

    # 4. Sessões inválidas: duração negativa ou zero substituída por null
    .withColumn(
        'tempo_pagina_seg',
        F.when(F.col('tempo_pagina_seg') > 0, F.col('tempo_pagina_seg')).otherwise(F.lit(None))
    )

    # 5. Colunas derivadas de tempo
    .withColumn('hora_evento',       F.hour('data_evento').cast('int'))
    .withColumn('dia_semana_evento', F.date_format('data_evento', 'EEEE'))
    .withColumn('ano_mes_evento',    F.date_format('data_evento', 'yyyy-MM'))

    # 6. Período do dia baseado na hora
    .withColumn(
        'periodo_dia',
        F.when(F.col('hora_evento').between(0,  5), 'Madrugada')
         .when(F.col('hora_evento').between(6, 11), 'Manha')
         .when(F.col('hora_evento').between(12, 17), 'Tarde')
         .otherwise('Noite')
    )

    # 7. Flag de conversão (evento que representa compra concluída)
    .withColumn(
        'is_conversao',
        F.col('tipo_evento') == 'Compra'
    )

    # 8. Marca temporal desta carga Silver
    .withColumn('timestamp_ingestion', F.current_timestamp())

    # 9. Ordenação de colunas do schema Silver
    .select(
        'id_evento',
        'id_sessao',
        'id_cliente',
        'id_dispositivo',
        'id_produto',
        'tipo_evento',
        'canal',
        'dispositivo',
        'origem_sessao',
        'data_evento',
        'hora_evento',
        'dia_semana_evento',
        'periodo_dia',
        'ano_mes_evento',
        'tempo_pagina_seg',
        'is_conversao',
        'timestamp_ingestion',
    )
)

print('Transformações aplicadas. Schema resultante:')
df_silver_cs.printSchema()

In [0]:
# ─── Validação pré-escrita ────────────────────────────────────────────────────

total_cs    = df_silver_cs.count()
conversoes  = df_silver_cs.filter(F.col('is_conversao') == True).count()
sem_cliente = df_silver_cs.filter(F.col('id_cliente').isNull()).count()
sem_duracao = df_silver_cs.filter(F.col('tempo_pagina_seg').isNull()).count()
outros_tipo = df_silver_cs.filter(F.col('tipo_evento') == 'Outro').count()

print(f'Total de eventos           : {total_cs:,}')
print(f'Conversoes (Compra)        : {conversoes:,} ({conversoes/total_cs*100:.1f}%)')
print(f'Sessoes sem duracao valida : {sem_duracao:,} ({sem_duracao/total_cs*100:.1f}%)')
print(f'Clientes anonimos (null)   : {sem_cliente:,} ({sem_cliente/total_cs*100:.1f}%)')
print(f'tipo_evento "Outro"        : {outros_tipo:,}')
print()
print('Distribuicao de tipo_evento apos normalizacao:')
df_silver_cs.groupBy('tipo_evento').count().orderBy(F.desc('count')).show()
print('Distribuicao de dispositivo:')
df_silver_cs.groupBy('dispositivo').count().orderBy(F.desc('count')).show()
print('Distribuicao de origem_sessao:')
df_silver_cs.groupBy('origem_sessao').count().orderBy(F.desc('count')).show()
print('Distribuicao de canal:')
df_silver_cs.groupBy('canal').count().orderBy(F.desc('count')).show()
print('Distribuicao de periodo_dia:')
df_silver_cs.groupBy('periodo_dia').count().orderBy(F.desc('count')).show()

In [0]:
# ─── Grava silver.ft_clickstream ────────────────────────────────────────────
# overwrite garante idempotência: reexecutar o notebook produz o mesmo resultado.

(
    df_silver_cs.write
                .format('delta')
                .mode('overwrite')
                .option('overwriteSchema', 'true')
                .saveAsTable(f'{silver_schema}.ft_clickstream')
)

print(f'{silver_schema}.ft_clickstream gravada com {df_silver_cs.count():,} registros.')

In [0]:
# ─── Resumo final ────────────────────────────────────────────────────────────

tabelas = [
    f'{silver_schema}.ft_tickets_suporte',
    f'{silver_schema}.dim_tipos_problema',
    f'{silver_schema}.dim_agentes_suporte',
    f'{silver_schema}.ft_clickstream',
]

print('=== Camada Silver — Resumo ===')
print(f'{"Tabela":<40} {"Linhas":>8} {"Colunas":>8}')
print('-' * 60)
for t in tabelas:
    df_t = spark.table(t)
    print(f'{t:<40} {df_t.count():>8,} {len(df_t.columns):>8}')

---

## Tratamento: `ft_avaliacoes`

**Origem:** `bronze.avaliacoes`  
**Destino:** `silver.ft_avaliacoes`

### Problemas identificados na Bronze

| Coluna | Problema | Tratamento |
|---|---|---|
| `nota_produto` | Valores fora do range 1–5 (`-1`, `0`, `6`) e textos (`bom`, `ruim`, `péssimo`, `ótimo`) | Mapear textos para numérico; anular valores fora de 1–5 com `null` |
| `nota_nps` | Valores fora do range 0–10 (`-1`, `11`) e mesmos textos acima | Mapear textos para numérico; anular valores fora de 0–10 com `null` |
| `recomenda` | ~12 variações (`S`, `sim`, `SIM`, `yes`, `1` / `N`, `Nao`, `NAO`, `no`, `0`) | Normalizar para booleano (`true` / `false`) |
| `data_avaliacao` | 3.174 registros nulos; formato `datetime com espaço` | Cast para `timestamp`; nulos mantidos como `null` |
| Duplicatas | Possíveis reprocessamentos da Bronze (`mode=append`) | Deduplicação por `id_avaliacao` |

### Colunas derivadas criadas

| Coluna | Lógica |
|---|---|
| `categoria_nps` | Detrator (0–6) · Neutro (7–8) · Promotor (9–10) — `null` quando `nota_nps` é inválida |
| `recomenda` | Convertido de texto/número para boolean |


In [0]:
# ─── Leitura da Bronze ────────────────────────────────────────────────────────
# Lê todos os registros brutos da tabela de avaliações ingerida na camada Bronze.
# Como a Bronze usa mode=append, reprocessamentos podem gerar registros duplicados
# para o mesmo id_avaliacao. Usamos row_number() sobre a janela particionada por
# id_avaliacao (ordenada pelo timestamp_ingestion mais recente) para ficar apenas
# com a versão mais atual de cada avaliação.

df_raw = spark.table(f'{bronze_schema}.avaliacoes')

window_dedup = Window.partitionBy('id_avaliacao').orderBy(F.desc('timestamp_ingestion'))

df_dedup = (
    df_raw
    .withColumn('_rank', F.row_number().over(window_dedup))
    .filter(F.col('_rank') == 1)
    .drop('_rank', 'timestamp_ingestion')
)

total_raw = df_raw.count()
total_dedup = df_dedup.count()

print(f'Registros na Bronze  : {total_raw:,}')
print(f'Após deduplicação    : {total_dedup:,}')
print(f'Duplicatas removidas : {total_raw - total_dedup:,}')


In [0]:
# ─── Mapeamento de notas textuais -> numéricas ────────────────────────────────
# Na Bronze, tanto nota_produto quanto nota_nps apresentam valores textuais
# misturados com valores numéricos. Os textos encontrados e seus equivalentes
# numéricos adotados são:
#
#   'ótimo'   -> 5   (nota máxima)
#   'bom'     -> 4   (nota boa)
#   'ruim'    -> 2   (nota ruim)
#   'péssimo' -> 1   (nota mínima)
#
# Após a conversão textual, valores fora dos ranges válidos são anulados:
#   nota_produto : range esperado 1–5  → fora disso vira null
#   nota_nps     : range esperado 0–10 → fora disso vira null

texto_para_nota = {
    'ótimo':   5,
    'bom':     4,
    'ruim':    2,
    'péssimo': 1,
}
# Constrói expressão que converte texto para número antes do cast final.
# Para cada par (texto, valor) no dicionário, encadeia um WHEN; o OTHERWISE
# mantém o valor original (caso já seja numérico em string).

def texto_para_numero_da_nota(coluna: str):
    # Essa função retorna uma coluna Pyspark que normaliza os textos para numero int
    expressao = F.col(coluna)
    for texto, nota in texto_para_nota.items():
        # Cast final para int e valores que nao sao numericos que sobrarem viram null
        expressao = F.when(F.lower(F.col(coluna)) == texto, F.lit(nota)).otherwise(expressao)
    return expressao.cast('int')

print('Expressões de mapeamento criadas.')
print(f'Textos mapeados: {list(texto_para_nota.keys())}')

In [0]:
# ─── Mapeamento de recomenda -> boolean ───────────────────────────────────────
# O campo recomenda chegou com ~12 variações para dois valores semânticos:
#
#   Positivo (true)  : 'S', 'Sim', 'SIM', 'sim', 'yes', '1'
#   Negativo (false) : 'N', 'Nao', 'NAO', 'nao', 'no',  '0'
#
# A estratégia é normalizar para lowercase e checar pertencimento ao conjunto
# positivo. Qualquer valor não reconhecido vira null para não inferir intenção.

valor_recomenda_positivo = {'s', 'sim', 'yes', '1'}
valor_recomenda_negativo = {'n', 'nao', 'no', '0'}

expressao_recomenda = (
    F.when(F.lower(F.col('recomenda')).isin(valor_recomenda_positivo), True)
    .when(F.lower(F.col('recomenda')).isin(valor_recomenda_negativo), False)
    .otherwise(None) #aqui é para os valores que a gente nao conhece ai não inferimos
    .cast('boolean')
)

print('Expressão de normalização de recomenda criada.')
print(f'  Positivo (true) : {valor_recomenda_positivo}')
print(f'  Negativo (false): {valor_recomenda_negativo}')

In [0]:
# ─── Transformações principais ───────────────────────────────────────────────
# Aplica sequencialmente todos os tratamentos definidos acima sobre o DataFrame
# deduplicado, produzindo o DataFrame Silver final.

df_silver = (
    df_dedup

    # 1. Converte notas textuais para inteiro e anula valores fora do range válido
    #    nota_produto: range 1–5 (escala de satisfação do produto)
    #    nota_nps: range 0–10 (Net Promoter Score)
    .withColumn('nota_produto',
        F.when(
            texto_para_numero_da_nota('nota_produto').between(1, 5),
            texto_para_numero_da_nota('nota_produto')
        ).otherwise(F.lit(None).cast('int'))   # fora do range -> null
    )
    .withColumn('nota_nps',
        F.when(
            texto_para_numero_da_nota('nota_nps').between(0, 10),
            texto_para_numero_da_nota('nota_nps')
        ).otherwise(F.lit(None).cast('int'))   # fora do range -> null
    )

    # 2. Normaliza recomenda para boolean usando a expressão construída acima
    .withColumn('recomenda', expressao_recomenda)

     # 3. Cast de data_avaliacao para timestamp
    #    Dois formatos coexistem na Bronze:
    #      - Padrão  : 'yyyy-MM-dd HH:mm:ss'  -> maioria dos registros
    #      - Variante: 'dd/MM/yyyy HH:mm:ss'  -> ~12.5k registros com barras
    #    try_to_timestamp é usado no lugar de to_timestamp pois nunca lança
    #    exceção — retorna null quando o valor não bate com o formato, o que
    #    torna o coalesce seguro mesmo com dados malformados.
    .withColumn('data_avaliacao', 
        F.coalesce(
            # Tenta o formato padrão com hífen (Ano-Mês-Dia)
            F.expr("try_to_timestamp(data_avaliacao, 'yyyy-MM-dd HH:mm:ss')"),    
            # Tenta o formato com barras (Ano/Dia/Mês) - O primeiro que deu erro
            F.expr("try_to_timestamp(data_avaliacao, 'yyyy/dd/MM HH:mm:ss')"),
            # Tenta o formato Brasileiro com barras (Dia/Mês/Ano) - Visto no print
            F.expr("try_to_timestamp(data_avaliacao, 'dd/MM/yyyy HH:mm:ss')"), 
            # Tenta o formato Americano com hifens (Mês-Dia-Ano) - Visto no print
            F.expr("try_to_timestamp(data_avaliacao, 'MM-dd-yyyy HH:mm:ss')"),
            # Fallback nativo do Spark para tentar salvar o que sobrar
            F.expr("try_to_timestamp(data_avaliacao)")
        )
    )

    # 4. Categoria NPS derivada — classificação padrão de mercado:
    #      0–6  -> Detrator  (clientes insatisfeitos, risco de churn)
    #      7–8  -> Neutro    (clientes passivos, sem engajamento forte)
    #      9–10 -> Promotor  (clientes leais, propensos a indicar)
    #    Quando nota_nps é null (valor inválido na origem), categoria_nps
    #    também fica null para não distorcer análises de NPS.
    .withColumn('categoria_nps',
        F.when(F.col('nota_nps').between(0, 6),  'Detrator')
         .when(F.col('nota_nps').between(7, 8),  'Neutro')
         .when(F.col('nota_nps').between(9, 10), 'Promotor')
         .otherwise(None)
    )

    # 5. Marca temporal desta carga Silver (recriado para rastreabilidade)
    .withColumn('timestamp_ingestion', F.current_timestamp())

    # 6. Ordenação de colunas conforme schema Silver acordado
    .select(
        'id_avaliacao',
        'id_pedido',
        'id_cliente',
        'id_produto',
        'nota_produto',
        'comentario',
        'nota_nps',
        'categoria_nps',
        'recomenda',
        'data_avaliacao',
        'timestamp_ingestion',
    )
)

print('Transformações aplicadas. Schema resultante:')
df_silver.printSchema()

In [0]:
# ─── Validação pré-escrita ────────────────────────────────────────────────────
# Antes de gravar, verificamos as principais métricas de qualidade para garantir
# que os tratamentos produziram o resultado esperado. Qualquer número suspeito
# deve ser investigado antes de prosseguir.

total = df_silver.count()

# Notas que viraram null após tratamento (eram inválidas na origem)
notas_produto_nulas = df_silver.filter(F.col('nota_produto').isNull()).count()
notas_nps_nulas = df_silver.filter(F.col('nota_nps').isNull()).count()
recomenda_nulas = df_silver.filter(F.col('recomenda').isNull()).count()
datas_nulas = df_silver.filter(F.col('data_avaliacao').isNull()).count()

print(f'Total de avaliações: {total:,}')
print()
print(f'nota_produto nulas (inválidas): {notas_produto_nulas:,} ({notas_produto_nulas/total*100:.1f}%)')
print(f'nota_nps nulas (inválidas): {notas_nps_nulas:,} ({notas_nps_nulas/total*100:.1f}%)')
print(f'recomenda nulas: {recomenda_nulas:,} ({recomenda_nulas/total*100:.1f}%)')
print(f'data_avaliacao nulas: {datas_nulas:,} ({datas_nulas/total*100:.1f}%)')
print()
print('Distribuição de categoria_nps:')
df_silver.groupBy('categoria_nps').count().orderBy('categoria_nps').show()
print('Distribuição de recomenda:')
df_silver.groupBy('recomenda').count().orderBy('recomenda').show()
print('Distribuição de nota_produto (range 1–5):')
df_silver.groupBy('nota_produto').count().orderBy('nota_produto').show()

In [0]:
# ─── Grava silver.ft_avaliacoes ──────────────────────────────────────────────
# Usa mode=overwrite para garantir idempotência: reexecutar este notebook
# sempre produz o mesmo resultado, sem acumular registros duplicados.
# overwriteSchema=true permite que alterações de schema futuras não quebrem
# a execução.

df_silver.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable(f'{silver_schema}.ft_avaliacoes')

print(f'{silver_schema}.ft_avaliacoes gravadas com {df_silver.count():,} registros.')

In [0]:
# ─── Resumo final ─────────────────────────────────────────────────────────────
# Confirma a tabela gravada e exibe contagem e número de colunas,
# seguindo o mesmo padrão de log das outras seções deste notebook.

tabelas = [
    f'{silver_schema}.ft_avaliacoes',
]

print('=== Camada Silver — Avaliações Pós-Compra ===')
print(f'{"Tabela":<40} {"Linhas":>8} {"Colunas":>8}')
print('-' * 60)
for t in tabelas:
    df_t = spark.table(t)
    print(f'{t:<40} {df_t.count():>8,} {len(df_t.columns):>8}')

# Tratamento Silver — `dim_categorias_produto`

---

**Origem:** `bronze.catalogo_produtos`  
**Destino:** `silver.dim_categorias_produto`

---

## Problemas identificados na Bronze

| Coluna | Problema | Tratamento |
|---|---|---|
| `categoria` | **Aproximadamente 50 variações** para 8 categorias reais, distribuídas em: Variações de case (`VESTUARIO`, `eletronicos`), tipos com substituição de letras por números (`M0VEIS`, `Cas4`, `3sportes`, `automotiv3`, `B3LEZA`, `BR1NQUEDOS`), abreviações (`vest`, `elet`, `brin`, `cas`, `esp`, `aut`, `mov`, `bel`), acentuação inconsistente (`Eletrônico`, `Móveis`) e **27 nulos** | Normalização para lowercase + Mapeamento para 8 valores canônicos + Nulos descartados |

---

## Mapeamento canônico de `categoria`

As 8 categorias reais identificadas e todas as variações encontradas no dataset:

| Valor Canônico | Variações mapeadas |
|---|---|
| `Eletronicos` | `eletronicos`, `ELETRONICOS`, `Eletrônico`, `Eletronico`, `elet`, `ELET` |
| `Vestuario` | `VESTUARIO`, `vestuario`, `Vestuarios`, `vest`, `vestu`, `VEST` |
| `Moveis` | `moveis`, `MOVEIS`, `M0VEIS`, `Móveis`, `mov` |
| `Casa` | `casa`, `CASA`, `cas`, `cas@`, `Cas4` |
| `Esportes` | `esportes`, `ESPORTES`, `Esporte`, `esport`, `ESPORT`, `esp`, `3sportes` |
| `Beleza` | `beleza`, `BELEZA`, `B3LEZA`, `bel`, `Belz` |
| `Brinquedos` | `brinquedos`, `Brinquedo`, `BR1NQUEDOS`, `brin`, `Brinq` |
| `Automotivo` | `automotivo`, `Automotivo`, `automotiv3`, `aut`, `Autom` |

---

## Colunas derivadas criadas

| Coluna | Lógica |
|---|---|
| `segmento` | Agrupamento de negócio derivado da categoria canônica |

### Mapeamento de categorias para segmentos

| Categoria | Segmento |
|---|---|
| `Eletronicos` | Tecnologia |
| `Vestuario` | Moda |
| `Moveis` | Casa & Decoração |
| `Casa` | Casa & Decoração |
| `Esportes` | Bem-Estar |
| `Beleza` | Bem-Estar |
| `Brinquedos` | Entretenimento |
| `Automotivo` | Automotivo |

---

## Schema final — `silver.dim_categorias_produto`

| Coluna | Tipo | Observação |
|---|---|---|
| `categoria` | `string` | Valor canônico — chave desta dimensão; 1 registro por categoria |
| `segmento` | `string` | Agrupamento de negócio derivado |
| `timestamp_ingestion` | `timestamp` | Instante da carga Silver |

---

In [0]:
# ─── Leitura da Bronze ────────────────────────────────────────────────────────
# Usa a última ingestão de cada produto (maior timestamp_ingestion) para
# garantir que reprocessamentos da Bronze não gerem duplicatas na Silver.

df_raw = spark.table(f'{bronze_schema}.catalogo_produtos')

window_dedup = Window.partitionBy('id_produto').orderBy(F.desc('timestamp_ingestion'))

df_dedup = (
    df_raw
    .withColumn('_rank', F.row_number().over(window_dedup))
    .filter(F.col('_rank') == 1)
    .drop('_rank', 'timestamp_ingestion')   # será recriado com o instante desta carga
)

total_raw   = df_raw.count()
total_dedup = df_dedup.count()
print(f'Registros na Bronze  : {total_raw:,}')
print(f'Após deduplicação    : {total_dedup:,}')
print(f'Duplicatas removidas : {total_raw - total_dedup:,}')

In [0]:
# ─── Mapeamento canônico de categoria ────────────────────────────────────────
# Na Bronze foram encontradas 8 categorias reais fragmentadas em aproximadamente 50 variações:
#
#   Eletronicos -> eletronicos, ELETRONICOS, Eletrônico, Eletronico, elet, ELET
#   Vestuario   -> VESTUARIO, vestuario, Vestuarios, vest, vestu, VEST
#   Moveis      -> moveis, MOVEIS, M0VEIS, Móveis, mov
#   Casa        -> casa, CASA, cas, cas@, Cas4
#   Esportes    -> esportes, ESPORTES, Esporte, esport, ESPORT, esp, 3sportes
#   Beleza      -> beleza, BELEZA, B3LEZA, bel, Belz
#   Brinquedos  -> brinquedos, Brinquedo, BR1NQUEDOS, brin, Brinq
#   Automotivo  -> automotivo, Automotivo, automotiv3, aut, Autom
#
# Estratégia: Normaliza-se os valores para lowercase e aplicamos mapeamento por valor exato.
# Observação: Valores não reconhecidos e nulos são descartados — por se tratar de uma
# dimensão de referência, um valor desconhecido não deve entrar no catálogo canônico.

# Mapeamento das categorias:
categoria_map = {
    # Eletronicos
    'eletronicos' : 'Eletronicos', 'eletrônicos' : 'Eletronicos',
    'eletronico'  : 'Eletronicos', 'eletrônico'  : 'Eletronicos',
    'elet'        : 'Eletronicos', 'ELET'         : 'Eletronicos',
    # Vestuario
    'vestuario'   : 'Vestuario',   'vestuários'  : 'Vestuario',
    'vestuarios'  : 'Vestuario',   'vest'         : 'Vestuario',
    'vestu'       : 'Vestuario',
    # Moveis
    'moveis'      : 'Moveis',      'móveis'       : 'Moveis',
    'm0veis'      : 'Moveis',      'mov'           : 'Moveis',
    # Casa
    'casa'        : 'Casa',        'cas@'          : 'Casa',
    'cas4'        : 'Casa',        'cas'           : 'Casa',
    # Esportes
    'esportes'    : 'Esportes',    'esporte'       : 'Esportes',
    'esport'      : 'Esportes',    'esp'           : 'Esportes',
    '3sportes'    : 'Esportes',
    # Beleza
    'beleza'      : 'Beleza',      'b3leza'        : 'Beleza',
    'bel'         : 'Beleza',      'belz'          : 'Beleza',
    # Brinquedos
    'brinquedos'  : 'Brinquedos',  'brinquedo'    : 'Brinquedos',
    'br1nquedos'  : 'Brinquedos',  'brin'          : 'Brinquedos',
    'brinq'       : 'Brinquedos',
    # Automotivo
    'automotivo'  : 'Automotivo',  'automotiv3'   : 'Automotivo',
    'aut'         : 'Automotivo',  'autom'         : 'Automotivo',
}

# Constrói a expressão CASE WHEN a partir do dicionário.
# Normaliza para lowercase antes de comparar para cobrir variações de case.
categoria_expr = F.lit(None).cast('string')
for raw_val, canonical in categoria_map.items():
    categoria_expr = F.when(
        F.lower(F.trim(F.col('categoria'))) == raw_val.lower(), canonical
    ).otherwise(categoria_expr)

print(f'Expressão de mapeamento criada para {len(categoria_map)} variações.')

In [0]:
# ─── Transformações principais ───────────────────────────────────────────────
# Aplica-se o mapeamento canônico, descarta nulos e valores não reconhecidos.
# Deduplica para 1 registro por categoria e deriva o segmento de negócio.
#
# Segmentos derivados:
#   Eletronicos -> Tecnologia
#   Vestuario   -> Moda
#   Moveis      -> Casa & Decoração
#   Casa        -> Casa & Decoração
#   Esportes    -> Bem-Estar
#   Beleza      -> Bem-Estar
#   Brinquedos  -> Entretenimento
#   Automotivo  -> Automotivo

# Transformação principal:
df_silver = (
    df_dedup

    # 1. Aplica mapeamento canônico — valores não reconhecidos e nulos ficam 'null'
    .withColumn('categoria', categoria_expr)

    # 2. Descarta registros sem categoria canônica (nulos originais + não mapeados)
    .filter(F.col('categoria').isNotNull())

    # 3. Mantém apenas a coluna categoria e deduplica para 1 linha por categoria
    .select('categoria')
    .distinct()

    # 4. Deriva segmento de negócio a partir da categoria canônica
    .withColumn(
        'segmento',
        F.when(F.col('categoria') == 'Eletronicos', 'Tecnologia')
         .when(F.col('categoria') == 'Vestuario',   'Moda')
         .when(F.col('categoria') == 'Moveis',      'Casa & Decoração')
         .when(F.col('categoria') == 'Casa',        'Casa & Decoração')
         .when(F.col('categoria') == 'Esportes',    'Bem-Estar')
         .when(F.col('categoria') == 'Beleza',      'Bem-Estar')
         .when(F.col('categoria') == 'Brinquedos',  'Entretenimento')
         .when(F.col('categoria') == 'Automotivo',  'Automotivo')
    )

    # 5. Marca temporal de ingestão
    .withColumn('timestamp_ingestion', F.current_timestamp())

    # 6. Ordenação das colunas
    .select(
        'categoria',
        'segmento',
        'timestamp_ingestion',
    )

    .orderBy('categoria')
)

print('Transformações aplicadas. Schema resultante:')
df_silver.printSchema()

In [0]:
# ─── Validação pré-escrita ────────────────────────────────────────────────────
# Confirmação do resultado para ter certeza que a transformação está correta.
# Espera-se exatamente 8 registros — um por categoria canônica — e que nenhum valor de segmento ficou nulo.

total            = df_silver.count()
segmentos_nulos  = df_silver.filter(F.col('segmento').isNull()).count()

print(f'Total de categorias canônicas : {total}')
print(f'Segmentos nulos (esperado = 0): {segmentos_nulos}')
print()
print('Conteúdo completo da dimensão:')
df_silver.show(truncate=False)

In [0]:
# ─── Grava silver.dim_categorias_produto ─────────────────────────────────────
# Usa-se overwrite para garantir idempotência: A reexecução do notebook produz o mesmo resultado.

(
    df_silver.write
             .format('delta')
             .mode('overwrite')
             .option('overwriteSchema', 'true')
             .saveAsTable(f'{silver_schema}.dim_categorias_produto')
)

print(f'{silver_schema}.dim_categorias_produto gravada com {df_silver.count()} registros.')

In [0]:
# ─── Resumo final ─────────────────────────────────────────────────────────────

tabelas = [
    f'{silver_schema}.dim_categorias_produto',
]

print('=== Camada Silver — Categorias de Produto ===')
print(f'{"Tabela":<45} {"Linhas":>8} {"Colunas":>8}')
print('-' * 65)
for t in tabelas:
    df_t = spark.table(t)
    print(f'{t:<45} {df_t.count():>8,} {len(df_t.columns):>8}')

## Tratamento: `dim_clientes`

**Origem:** `bronze.clientes`  
**Destino:** `silver.dim_clientes`

### Problemas identificados na Bronze

| Coluna | Problema | Tratamento |
|---|---|---|
| `nome`, `sobrenome` | Variações de case (`JOAO`, `joao`, `João`) | `INITCAP()` + `TRIM()` |
| `email` | Case misto (`Joao@Gmail.com`) | `LOWER()` + `TRIM()` |
| `genero` | ~10 variações para 3 categorias (`M`, `male`, `MASC`…) | Mapeamento canônico via cadeia de `WHEN` |
| `origem` | Variações (`app`, `APP`, `App Mobile`, `web`, `WEB`…) | `LOWER()` + `TRIM()` + mapeamento canônico |
| `cidade` | Case misto (`são paulo`, `SÃO PAULO`) | `INITCAP()` + `TRIM()` |
| `estado` | Case misto e abreviações inconsistentes | `UPPER()` + `TRIM()` |
| `pais` | Case misto | `UPPER()` + `TRIM()` |
| `data_nascimento`, `data_cadastro` | Inferidas como `string` na Bronze | Cast explícito para `date` |
| `telefone`, `endereco` | Possíveis espaços extras | `TRIM()` |
| `device_ids` | String bruta — possível array serializado | Mantido como `string` para análise futura |
| Duplicatas | Possíveis reprocessamentos da Bronze (`mode=append`) | `ROW_NUMBER()` sobre `id_cliente` por `timestamp_ingestion DESC` |

### Colunas derivadas

| Coluna | Lógica |
|---|---|
| `nome_completo` | `CONCAT(nome, ' ', sobrenome)` |
| `idade` | `FLOOR(DATEDIFF(current_date, data_nascimento) / 365)` |
| `faixa_etaria` | `CASE` sobre `idade`: Menor de 18 / 18-24 / 25-34 / 35-44 / 45-59 / 60+ |
| `tempo_cliente_dias` | `DATEDIFF(current_date, data_cadastro)` |
| `regiao` | `CASE` sobre `estado` → Norte / Nordeste / Centro-Oeste / Sudeste / Sul |
| `timestamp_ingestion` | `current_timestamp()` — recriado no momento da carga Silver |

In [0]:
#Leitura da Bronze
#Usa o registro mais recente por id_cliente (maior timestamp_ingestion) para
#Neutralizar reprocessamentos da Bronze em mode=append.

df_raw = spark.table(f'{bronze_schema}.clientes')

window_dedup = Window.partitionBy('id_cliente').orderBy(F.desc('timestamp_ingestion'))

df_dedup = (
    df_raw
    .withColumn('_rank', F.row_number().over(window_dedup))
    .filter(F.col('_rank') == 1)
    .drop('_rank', 'timestamp_ingestion')   #remove timestamp_ingestion que será recriado ao final
)

total_raw   = df_raw.count()
total_dedup = df_dedup.count()
print(f'Registros na Bronze  : {total_raw:,}')
print(f'Após deduplicação    : {total_dedup:,}')
print(f'Duplicatas removidas : {total_raw - total_dedup:,}')

In [0]:
#Mapeamentos canônicos
#Genero: 3 categorias reais fragmentadas em múltiplas variações
#Masculino  - m, masculino, male, masc, homem, h
#Feminino   - f, feminino, female, fem, mulher
#Outro      - outro, other, nb, nao_binario, nao informado
#Garantem que qualquer análise que for feita em cima de dados com esssas categorias sejam consistentes

genero_map = {
    'm'          : 'Masculino', 'masculino': 'Masculino', 'male' : 'Masculino',
    'masc'       : 'Masculino', 'homem'    : 'Masculino', 'h'    : 'Masculino',
    'f'          : 'Feminino',  'feminino' : 'Feminino',  'female': 'Feminino',
    'fem'        : 'Feminino',  'mulher'   : 'Feminino',
    'o'          : 'Outro',     'outro'    : 'Outro',     'other': 'Outro',
    'nb'         : 'Outro',     'nao_binario': 'Outro',
    'nao-informado' : 'Outro',
    'nao informado': 'Outro',
}

genero_expr = F.lit('Não informado')
for raw_val, canonical in genero_map.items():
    genero_expr = F.when(
        F.lower(F.trim(F.col('genero'))) == raw_val, canonical
    ).otherwise(genero_expr)

#origem: canônicas — App / Web / Parceiro / Não informado
origem_map = {
    'app'        : 'App',      'app mobile': 'App',    'mobile': 'App',
    'web'        : 'Web',      'website'   : 'Web',    'site'  : 'Web',
    'parceiro'   : 'Parceiro', 'partner'   : 'Parceiro',
    'indicação'  : 'Indicação', 'indicacao'  : 'Indicação', 'indicaçao': 'Indicação',
}

origem_expr = F.lit('Não informado')
for raw_val, canonical in origem_map.items():
    origem_expr = F.when(
        F.lower(F.trim(F.col('origem'))) == raw_val, canonical
    ).otherwise(origem_expr)

print('Expressões de mapeamento criadas.')

In [0]:
#Principais transformações

regiao_expr = (
    #criação de coluna de agrupamento por região 
    

    F.when(F.upper(F.col('estado')).isin(
        'AMAZONAS','PARÁ','ACRE','RONDÔNIA','RORAIMA','AMAPÁ','TOCANTINS'
    ), 'Norte')
     .when(F.upper(F.col('estado')).isin(
        'BAHIA','SERGIPE','ALAGOAS','PERNAMBUCO','PARAÍBA',
        'RIO GRANDE DO NORTE','CEARÁ','PIAUÍ','MARANHÃO'
    ), 'Nordeste')
     .when(F.upper(F.col('estado')).isin(
        'MATO GROSSO','MATO GROSSO DO SUL','GOIÁS','DISTRITO FEDERAL'
    ), 'Centro-Oeste')
     .when(F.upper(F.col('estado')).isin(
        'SÃO PAULO','RIO DE JANEIRO','MINAS GERAIS','ESPÍRITO SANTO'
    ), 'Sudeste')
     .when(F.upper(F.col('estado')).isin(
        'PARANÁ','SANTA CATARINA','RIO GRANDE DO SUL'
    ), 'Sul')
     .otherwise('Não identificada')
)

df_silver = (
    df_dedup

    #Sanitização textual
    .withColumn('nome',      F.initcap(F.trim(F.col('nome'))))
    .withColumn('sobrenome', F.initcap(F.trim(F.col('sobrenome'))))
    .withColumn('email',     F.lower(F.trim(F.col('email'))))
    .withColumn('telefone',  F.trim(F.col('telefone')))
    .withColumn('endereco',  F.trim(F.col('endereco')))
    .withColumn('cidade',    F.initcap(F.trim(F.col('cidade'))))
    .withColumn('estado',    F.upper(F.trim(F.col('estado'))))
    .withColumn('pais',      F.upper(F.trim(F.col('pais'))))

    #Mapeamentos canônicos
    .withColumn('genero', genero_expr)
    .withColumn('origem', origem_expr)

    #Tipagem explícita
    .withColumn('data_nascimento', F.col('data_nascimento').cast('date'))
    .withColumn('data_cadastro',   F.col('data_cadastro').cast('date'))

    #Colunas derivadas
    .withColumn('nome_completo',
        F.concat_ws(' ', F.col('nome'), F.col('sobrenome'))
    )
    .withColumn('idade',
                F.when(
        F.datediff(F.current_date(), F.col('data_nascimento')) < 0, None
    ).otherwise(
        F.floor(F.datediff(F.current_date(), F.col('data_nascimento')) / 365).cast('int')
    ))
    .withColumn('faixa_etaria',
        F.when(F.col('idade') < 18,  'Menor de 18')
         .when(F.col('idade') <= 24, '18-24')
         .when(F.col('idade') <= 34, '25-34')
         .when(F.col('idade') <= 44, '35-44')
         .when(F.col('idade') <= 59, '45-59')
         .otherwise('60+')
    )
    .withColumn('tempo_cliente_dias',
        F.datediff(F.current_date(), F.col('data_cadastro'))
    )
    .withColumn('regiao', regiao_expr)

    #Rastreabilidade
    .withColumn('timestamp_ingestion', F.current_timestamp())

    #Ordenação de colunas para facilitar visualização
    .select(
        'id_cliente',
        'nome',
        'sobrenome',
        'nome_completo',
        'email',
        'telefone',
        'genero',
        'data_nascimento',
        'idade',
        'faixa_etaria',
        'data_cadastro',
        'tempo_cliente_dias',
        'endereco',
        'cidade',
        'estado',
        'regiao',
        'pais',
        'device_ids',
        'origem',
        'timestamp_ingestion',
    )
)

print('Transformações aplicadas. Schema resultante:')
df_silver.printSchema()

In [0]:
#Validação pré-escrita 
#Exibir métricas do df antes de escrever no formato delta

total             = df_silver.count()
nulos_nascimento  = df_silver.filter(F.col('data_nascimento').isNull()).count()
nulos_cadastro    = df_silver.filter(F.col('data_cadastro').isNull()).count()
nao_informado_gen = df_silver.filter(F.col('genero') == 'Não informado').count()
nao_informado_ori = df_silver.filter(F.col('origem') == 'Não informado').count()

print(f'Total de clientes          : {total:,}')
print(f'Nulos em data_nascimento   : {nulos_nascimento:,}')
print(f'Nulos em data_cadastro     : {nulos_cadastro:,}')
print(f'Gênero "Não informado"     : {nao_informado_gen:,} ({nao_informado_gen/total*100:.1f}%)')
print(f'Origem "Não informado"     : {nao_informado_ori:,} ({nao_informado_ori/total*100:.1f}%)')
print()
print('Distribuição de gênero após normalização:')
df_silver.groupBy('genero').count().orderBy(F.desc('count')).show()
print('Distribuição de origem após normalização:')
df_silver.groupBy('origem').count().orderBy(F.desc('count')).show()
print('Distribuição de faixa_etaria:')
df_silver.groupBy('faixa_etaria').count().orderBy('faixa_etaria').show()

In [0]:
#Grava silver.dim_clientes
#overwrite garante idempotência: reexecutar o notebook produz o mesmo resultado.

(
    df_silver.write
             .format('delta')
             .mode('overwrite')
             .option('overwriteSchema', 'true')
             .saveAsTable(f'{silver_schema}.dim_clientes')
)

print(f'{silver_schema}.dim_clientes gravada com {df_silver.count():,} registros.')